## Overview of Architectural Conventions Applied

1. **Model & Tokenizer:** Uses `sarvamai/sarvam-2b-v0.5` (`LlamaForCausalLM` architecture with 28 layers, 2048 hidden size, and SentencePiece `▁` prefix handling) via `LlamaHelper`.


2. **Cloze Task Formulation:** Unlike translation prompts, the cloze task evaluates fill-in-the-blank / masked sentence prompts (using columns like `blank_prompt_translation_masked` from clean CSV files).


3. **Dual Trajectory Tracking:** Tracks both the **Latent Token Probability** (the English/source concept, `en`) and the **Output Token Probability** (the target regional language translation, `target_lang`) across all 28 layers using the logit lens.


4. **Intersection Disjoint Filter:** Filters out rows where latent token IDs overlap with target token IDs to prevent shared substrings/cognates from artificially inflating latent probabilities (with safety checks for Indic scripts).


5. **VRAM Optimization:** Slices only the last token before unembedding (`last_latents = latents[:, -1, :]`) to save ~95% VRAM during layer-wise logit lens extraction.

### Key Design Clarifications Before Code Execution

1. **Cloze Prompt Template:** Uses masked sentence templates (`blank_prompt_translation_masked`) to evaluate how early in the network layers the model transitions from the English concept latent to the target regional language output[cite: 2].
2. **Intersection Filtering Rule:** Disjoint filtering ensures that latent token IDs (`en`) and target token IDs (`target_lang`) do not overlap, avoiding spurious probability inflation[cite: 2].
3. **Sarvam-2B Compatibility:** Fully compatible with `sarvamai/sarvam-2b-v0.5` RMSNorm and unembedding projection `unemb = nn.Sequential(llama.model.model.norm, llama.model.lm_head)`[cite: 1].

### Block 1: GPU & Environment Validation

In [29]:
import os

os.environ["CUDA_VISIBLE_DEVICES"] = "0"  # Physical GPU to select

import torch

# Create a small tensor so this process uses the selected GPU
x = torch.empty(1, device="cuda")
torch.cuda.synchronize()

free, total = torch.cuda.mem_get_info(0)

print("Kernel PID:", os.getpid())
print("CUDA_VISIBLE_DEVICES:", os.environ["CUDA_VISIBLE_DEVICES"])
print("Visible GPU count:", torch.cuda.device_count())
print("GPU name:", torch.cuda.get_device_name(0))
print(f"Free VRAM: {free / 1024**3:.2f} GiB")

Kernel PID: 4102071
CUDA_VISIBLE_DEVICES: 0
Visible GPU count: 1
GPU name: NVIDIA L40S
Free VRAM: 33.41 GiB


In [30]:
!nvidia-smi --query-gpu=index,memory.used,memory.free --format=csv

index, memory.used [MiB], memory.free [MiB]
0, 18 MiB, 32242 MiB
1, 11278 MiB, 34211 MiB


### Block 2: Autoreload & Imports

In [31]:
%load_ext autoreload
%autoreload 2

import pandas as pd
import sys
import os
# Add the parent directory to sys.path where llamawrapper.py and utils.py are located
parent_dir = os.path.abspath(os.path.join(os.getcwd(), '../..'))
if parent_dir not in sys.path:
    sys.path.append(parent_dir)
from dataclasses import dataclass
import json
import numpy as np
from matplotlib import pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
from llamawrapper import load_unemb_only, LlamaHelper
from scipy.stats import bootstrap
from utils import plot_ci, plot_ci_plus_heatmap
from tqdm import tqdm

os.environ["CUDA_VISIBLE_DEVICES"] = "0"
# Fix random seed for reproducibility
seed = 42
np.random.seed(seed)
torch.manual_seed(seed)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


### Block 3: Parameters & Cloze Task Setup

In [32]:
target_lang = 'guj'  # e.g., 'pun', 'mar', 'ben', 'tel', 'tam'
model_size = '2b'
custom_model = 'sarvamai/sarvam-2b-v0.5'
out_dir = './visuals'
hf_token = None  # Or your HF token string if needed

print(f"Running Cloze task for target language: {target_lang} using model {custom_model}")

Running Cloze task for target language: guj using model sarvamai/sarvam-2b-v0.5


### Block 4: Load Model & Unembedding Layer

In [33]:
llama = LlamaHelper(dir=custom_model, load_in_8bit=True, hf_token=hf_token)
tokenizer = llama.tokenizer
model = llama.model

unemb = nn.Sequential(llama.model.model.norm, llama.model.lm_head)
print(unemb)

# Prepare for energy plots
U = list(unemb[1].parameters())[0].detach().cpu().float()
weights = list(unemb[0].parameters())[0].detach().cpu().float()
print(f'U {U.shape} weights {weights.unsqueeze(0).shape}')
U_weighted = U.clone() 
U_weighted *= weights.unsqueeze(0)
U_normalized = U_weighted / ((U_weighted**2).sum(dim=1, keepdim=True))**0.5
v = U.shape[0]
TT = U_normalized.T @ U_normalized
avgUU = (((U_normalized.T @ U_normalized)**2).sum() / v**2)**0.5
print("avgUU:", avgUU.item())

Loading weights: 100%|██████████████████████████████████████████████| 255/255 [00:01<00:00, 231.87it/s]


Sequential(
  (0): LlamaRMSNorm((2048,), eps=1e-05)
  (1): Linear(in_features=2048, out_features=64128, bias=False)
)
U torch.Size([64128, 2048]) weights torch.Size([1, 2048])
avgUU: 0.04994095489382744


### Block 5: Token Prefix & Vocabulary Lookup Helpers

In [34]:
def token_prefixes(token_str: str):
    token_str = str(token_str).strip()
    n = len(token_str)
    tokens = [token_str[:i] for i in range(1, n+1)]
    return tokens 

def add_spaces(tokens):
    return ['▁' + t for t in tokens] + tokens

def capitalizations(tokens):
    return list(set(tokens))

def process_tokens(token_str: str, tokenizer, lang):
    with_prefixes = token_prefixes(token_str)
    with_spaces = add_spaces(with_prefixes)
    with_capitalizations = capitalizations(with_spaces)
    final_tokens = []
    vocab = tokenizer.get_vocab()
    for tok in with_capitalizations:
        if tok in vocab:
            final_tokens.append(vocab[tok])
    if len(final_tokens) == 0:
        encoded = tokenizer.encode(str(token_str).strip(), add_special_tokens=False)
        if len(encoded) > 0:
            final_tokens.append(encoded[0])
    return list(set(final_tokens))

id2voc = {id: voc for voc, id in llama.tokenizer.get_vocab().items()}
def get_tokens(token_ids, id2voc=id2voc):
    return [id2voc[tokid] for tokid in token_ids]

def compute_entropy(probas):
    return (-probas * torch.log2(probas + 1e-12)).sum(dim=-1)

### Block 6: Load Cloze Dataset & Apply Intersection Filtering

In [35]:
prefix = "./data/langs/"
df_cloze = pd.read_csv(f'{prefix}{target_lang}/clean.csv')

# Ensure the required cloze column exists
key = "blank_prompt_translation_masked"
if key not in df_cloze.columns:
    raise ValueError(f"Column '{key}' not found in {target_lang}/clean.csv. Ensure your dataset supports the cloze task format.")

n_skip = 2
dataset_gap = []

for idx, (idx_df, row) in tqdm(enumerate(df_cloze.iterrows()), total=len(df_cloze)):
    indices = set(list(range(len(df_cloze)))) - set([idx])
    if len(indices) < n_skip:
        continue
    idx_examples = np.random.choice(list(indices), n_skip, replace=False)
    
    prompt_template = f"{df_cloze.loc[idx_examples[0], key]}\n"
    prompt_template += f"{df_cloze.loc[idx_examples[1], key]}\n"

    out_token_str = str(row['word_translation'])
    latent_token_str = str(row['word_original'])  # English word
    
    out_token_id = process_tokens(out_token_str, tokenizer, target_lang)
    latent_token_id = process_tokens(latent_token_str, tokenizer, 'en')
    
    intersection = set(out_token_id).intersection(set(latent_token_id))
    if len(out_token_id) == 0 or len(latent_token_id) == 0:
        continue
    if target_lang != 'en' and len(intersection) > 0:
        continue 
        
    prompt = row[key].split(":")[0] + ": \""
    
    dataset_gap.append({
        'prompt': prompt_template + prompt,
        'out_token_id': out_token_id,
        'out_token_str': out_token_str,
        'latent_token_id': latent_token_id,
        'latent_token_str': latent_token_str,
    })

print(f"Valid cloze samples generated: {len(dataset_gap)}")

100%|████████████████████████████████████████████████████████████████| 132/132 [00:06<00:00, 21.15it/s]

Valid cloze samples generated: 132


### Block 7: Save Cloze Dataset

In [36]:
df_gap = pd.DataFrame(dataset_gap)
save_path = f'{os.path.join(out_dir, custom_model)}/cloze'
os.makedirs(save_path, exist_ok=True)
df_gap.to_csv(f'{save_path}/{target_lang}_dataset.csv', index=False)
df_gap.head()

,prompt,out_token_id,out_token_str,latent_token_id,latent_token_str
0,"નવ પછી આવતી સંખ્યા ""___"" છે. ઉત્તર: ""દસ"".\n""__...","[21963, 8843, 491, 6905, 3065, 63675]",પુસ્તક,"[9378, 5670, 428, 13427, 63575, 6396]",book
1,"""___"" એ ૧૦ દશકની બરાબર સંખ્યા છે. ઉત્તર: ""સો""....","[545, 645, 63628, 11061, 37848, 2265]",વાદળ,"[1219, 30149, 63531, 408, 1531]",cloud
2,"""___"" એ એવી જગ્યા છે જ્યાં આપણે ચાલીએ અને ઘર બ...","[63769, 5884, 724, 2183]",બેગ,"[28320, 428, 40413, 63575]",bag
3,"""___"" એ અવકાશમાં એક ચોક્કસ સ્થાન છે. ઉત્તર: ""બ...","[63602, 2547, 3386, 717]",મોં,"[19142, 48119, 442, 63548, 20158]",mouth
4,"""___"" એ એવો બિંદુ છે જે બંને બાજુઓથી સમાન અંતર...","[54980, 459, 717, 879, 63602]",માટી,"[11304, 1965, 403, 1621, 63516]",soil


### Block 8: Logit Lens Extraction Across All Layers 

In [ ]:
latent_token_probs = []
out_token_probs = []
entropy = []
energy = [] 
latents_all = []

torch.cuda.empty_cache()

with torch.no_grad():
    for idx, d in tqdm(enumerate(dataset_gap), total=len(dataset_gap)):
        prompt = d['prompt']
        latents = llama.latents_all_layers(prompt)
        
        # Slice last token before unembedding to save VRAM
        last_latents = latents[:, -1, :]
        del latents
        
        logits = unemb(last_latents.to(next(unemb.parameters()).device))
        last = logits.float().softmax(dim=-1).detach().cpu()
        del logits
        
        latent_token_probs.append(last[:, torch.tensor(d['latent_token_id'])].sum(dim=-1))
        out_token_probs.append(last[:, torch.tensor(d['out_token_id'])].sum(dim=-1))
        entropy.append(compute_entropy(last))
        
        last_latents_cpu = last_latents.float().detach().cpu()
        del last_latents
        latents_all.append(last_latents_cpu.clone())
        
        latents_normalized = last_latents_cpu / (((last_latents_cpu**2).mean(dim=-1, keepdim=True))**0.5)
        latents_normalized /= (latents_normalized.norm(dim=-1, keepdim=True))
        norm = ((U_normalized @ latents_normalized.T)**2).mean(dim=0)**0.5
        energy.append(norm / avgUU)

latent_token_probs = torch.stack(latent_token_probs)
out_token_probs = torch.stack(out_token_probs)
entropy = torch.stack(entropy)
energy = torch.stack(energy)
latents = torch.stack(latents_all)

print("Latents shape:", latents.shape)

 14%|█████████▎                                                       | 19/132 [00:02<00:12,  8.93it/s]

### Block 9: Plotting Probability Curves & Heatmap

In [ ]:
size2tik = {'2b': 4, '7b': 5, '13b': 5, '70b': 10}
tik_step = size2tik.get(model_size, 4)
num_layers = entropy.shape[1]

fig, ax, ax2 = plot_ci_plus_heatmap(
    latent_token_probs, 
    entropy, 
    'en (latent)', 
    color='tab:orange', 
    tik_step=tik_step, 
    do_colorbar=True,
    nums=[.99, 0.18, 0.025, 0.6]
)

if target_lang != 'en':
    plot_ci(
        ax2, 
        out_token_probs, 
        f'{target_lang} (target)', 
        color='tab:blue', 
        do_lines=False, 
        tik_step=tik_step
    )

ax2.set_xlabel('layer')
ax2.set_ylabel('probability')
ax2.set_xlim(0, num_layers + 1)
ax2.set_ylim(0, 1)
ax2.legend(loc='upper left')

plt.savefig(
    f'{save_path}/{model_size}_{target_lang}_probas_ent.pdf', 
    dpi=300, 
    bbox_inches='tight'
)
plt.show()

### Block 10: Energy Plot & Saving Tensors

In [ ]:
fig, ax2 = plt.subplots(figsize=(5, 3))
plot_ci(ax2, energy, 'energy', color='tab:green', do_lines=True, tik_step=tik_step)
ax2.set_xlabel('layer')
ax2.set_ylabel('energy')
ax2.set_xlim(0, num_layers + 1)

plt.savefig(f'{save_path}/{model_size}_{target_lang}_energy.pdf', dpi=300, bbox_inches='tight')
plt.show()

torch.save(latents, f'{save_path}/{model_size}_{target_lang}_latents.pt')
torch.save(latent_token_probs, f'{save_path}/{model_size}_{target_lang}_latent_probs.pt')
torch.save(out_token_probs, f'{save_path}/{model_size}_{target_lang}_out_probs.pt')

### Block 11: Cloze Task Evaluation & Accuracy Inspection

In [ ]:
import pandas as pd
import torch

# Patch get_logits for reliable device placement
embed_device = llama.model.model.embed_tokens.weight.device
def get_logits_fixed(prompt):
    inputs = llama.tokenizer(prompt, return_tensors="pt").to(embed_device)
    with torch.no_grad():
        logits = llama.model(inputs.input_ids).logits
        return logits
llama.get_logits = get_logits_fixed

results = []
correct_count = 0

print(f"Evaluating final-layer predictions for Cloze task ({target_lang})...\n")

with torch.no_grad():
    for idx, d in enumerate(dataset_gap):
        prompt = d["prompt"]
        expected_target_str = d["out_token_str"].strip()
        expected_token_ids = set(d["out_token_id"])

        logits = llama.get_logits(prompt)
        last_token_logits = logits[0, -1, :]

        top1_token_id = torch.argmax(last_token_logits).item()
        top1_token_str = (
            tokenizer.decode([top1_token_id])
            .strip()
            .replace("▁", "")
            .replace(" ", "")
        )

        is_token_id_match = top1_token_id in expected_token_ids
        is_exact_str_match = (top1_token_str == expected_target_str) or (
            expected_target_str.startswith(top1_token_str) and len(top1_token_str) > 0
        )

        is_correct = is_token_id_match or is_exact_str_match
        if is_correct:
            correct_count += 1

        target_prob_final = out_token_probs[idx, -1].item() if len(out_token_probs) > idx else 0.0

        results.append({
            "Index": idx,
            "Expected": expected_target_str,
            "Model Output": top1_token_str,
            "Is Correct": "✅ Yes" if is_correct else "❌ No",
            "Target Prob (L28)": f"{target_prob_final:.4f}"
        })

eval_df = pd.DataFrame(results)
accuracy = (correct_count / len(dataset_gap)) * 100

print("=" * 60)
print(f"CLOZE TASK EVALUATION ({target_lang})")
print(f"Total Evaluated Samples : {len(dataset_gap)}")
print(f"Correct Top-1 Predictions: {correct_count} / {len(dataset_gap)}")
print(f"Top-1 Greedy Accuracy   : {accuracy:.2f}%")
print("=" * 60)
display(eval_df.head(15))